# 📊 Exploração e Análise dos Dados de Dengue

## Objetivo
Este notebook tem como objetivo realizar a exploração inicial dos dados de dengue, clima e saneamento, entendendo a estrutura dos dados, identificando padrões e preparando insights para as próximas etapas.

## Dataset
- **Fonte**: Dados de dengue, clima e saneamento (2014-2025)
- **Granularidade**: Estado (UF) e Mensal
- **Variável Target**: Quantidade de Casos de Dengue

In [ ]:
# Importação das bibliotecas necessárias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

warnings.filterwarnings('ignore')

# Configurações de visualização
plt.style.use('default')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print("✅ Bibliotecas carregadas com sucesso!")

In [ ]:
# Carregamento dos dados
df = pd.read_csv('dados_dengue_clima_saneamento_2014_2025.csv')

print(f"📋 Informações básicas do dataset:")
print(f"   • Linhas: {df.shape[0]:,}")
print(f"   • Colunas: {df.shape[1]}")
print(f"   • Período: {df['Ano'].min()} - {df['Ano'].max()}")
print(f"   • Estados únicos: {df['COD_UF'].nunique()}")

df.head()

In [ ]:
# Informações detalhadas sobre o dataset
print("🔍 Informações detalhadas:")
print(df.info())

print("\n📈 Estatísticas descritivas:")
df.describe()

In [ ]:
# Verificação de valores ausentes
missing_values = df.isnull().sum()
missing_percent = (missing_values / len(df)) * 100

missing_df = pd.DataFrame({
    'Coluna': missing_values.index,
    'Valores_Ausentes': missing_values.values,
    'Percentual': missing_percent.values
}).sort_values('Valores_Ausentes', ascending=False)

print("❓ Análise de valores ausentes:")
print(missing_df[missing_df['Valores_Ausentes'] > 0])

# Visualização dos valores ausentes
if missing_df['Valores_Ausentes'].sum() > 0:
    plt.figure(figsize=(12, 6))
    sns.heatmap(df.isnull(), cbar=True, yticklabels=False, cmap='viridis')
    plt.title('Mapa de Valores Ausentes')
    plt.show()
else:
    print("✅ Nenhum valor ausente encontrado!")

In [ ]:
# Mapeamento dos códigos UF para nomes dos estados
uf_mapping = {
    'RO': 'Rondônia', 'AC': 'Acre', 'AM': 'Amazonas', 'RR': 'Roraima',
    'PA': 'Pará', 'AP': 'Amapá', 'TO': 'Tocantins', 'MA': 'Maranhão',
    'PI': 'Piauí', 'CE': 'Ceará', 'RN': 'Rio Grande do Norte', 'PB': 'Paraíba',
    'PE': 'Pernambuco', 'AL': 'Alagoas', 'SE': 'Sergipe', 'BA': 'Bahia',
    'MG': 'Minas Gerais', 'ES': 'Espírito Santo', 'RJ': 'Rio de Janeiro',
    'SP': 'São Paulo', 'PR': 'Paraná', 'SC': 'Santa Catarina',
    'RS': 'Rio Grande do Sul', 'MS': 'Mato Grosso do Sul', 'MT': 'Mato Grosso',
    'GO': 'Goiás', 'DF': 'Distrito Federal'
}

df['Estado'] = df['COD_UF'].map(uf_mapping)
df['Data'] = pd.to_datetime(df[['Ano', 'Mês']].assign(day=1))

print("✅ Colunas de Estado e Data criadas!")
print(f"Estados no dataset: {sorted(df['Estado'].unique())}")

In [ ]:
# Análise da variável target (Quantidade de Casos)
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Distribuição dos casos
axes[0, 0].hist(df['Quantidade de Casos'], bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Distribuição dos Casos de Dengue')
axes[0, 0].set_xlabel('Quantidade de Casos')
axes[0, 0].set_ylabel('Frequência')

# Box plot dos casos
axes[0, 1].boxplot(df['Quantidade de Casos'])
axes[0, 1].set_title('Box Plot - Casos de Dengue')
axes[0, 1].set_ylabel('Quantidade de Casos')

# Log dos casos (para melhor visualização)
axes[1, 0].hist(np.log1p(df['Quantidade de Casos']), bins=50, edgecolor='black', alpha=0.7)
axes[1, 0].set_title('Distribuição Log dos Casos de Dengue')
axes[1, 0].set_xlabel('Log(Casos + 1)')
axes[1, 0].set_ylabel('Frequência')

# Casos por ano
casos_por_ano = df.groupby('Ano')['Quantidade de Casos'].sum()
axes[1, 1].plot(casos_por_ano.index, casos_por_ano.values, marker='o')
axes[1, 1].set_title('Total de Casos por Ano')
axes[1, 1].set_xlabel('Ano')
axes[1, 1].set_ylabel('Total de Casos')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print(f"📊 Estatísticas da variável target:")
print(f"   • Média: {df['Quantidade de Casos'].mean():.2f}")
print(f"   • Mediana: {df['Quantidade de Casos'].median():.2f}")
print(f"   • Desvio Padrão: {df['Quantidade de Casos'].std():.2f}")
print(f"   • Mínimo: {df['Quantidade de Casos'].min()}")
print(f"   • Máximo: {df['Quantidade de Casos'].max()}")

In [ ]:
# Top 10 estados com mais casos
casos_por_estado = df.groupby('Estado')['Quantidade de Casos'].sum().sort_values(ascending=False)

plt.figure(figsize=(12, 8))
casos_por_estado.head(10).plot(kind='bar', color='steelblue')
plt.title('Top 10 Estados com Mais Casos de Dengue (2014-2025)')
plt.xlabel('Estado')
plt.ylabel('Total de Casos')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("🏆 Top 5 Estados com mais casos:")
for i, (estado, casos) in enumerate(casos_por_estado.head().items(), 1):
    print(f"   {i}. {estado}: {casos:,} casos")

In [ ]:
# Sazonalidade: Casos por mês
casos_por_mes = df.groupby('Mês')['Quantidade de Casos'].mean().sort_index()
meses_nomes = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun',
               'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']

plt.figure(figsize=(12, 6))
plt.plot(casos_por_mes.index, casos_por_mes.values, marker='o', linewidth=2, markersize=8)
plt.title('Sazonalidade da Dengue - Média de Casos por Mês')
plt.xlabel('Mês')
plt.ylabel('Média de Casos')
plt.xticks(range(1, 13), meses_nomes)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("📅 Sazonalidade identificada:")
meses_maior_incidencia = casos_por_mes.nlargest(3)
for mes, casos in meses_maior_incidencia.items():
    print(f"   • {meses_nomes[mes-1]}: {casos:.0f} casos em média")

In [ ]:
# Análise das variáveis climáticas
variaveis_clima = ['precipitacao_media_mensal_uf', 'temp_max_media_mensal_uf',
                   'temp_min_media_mensal_uf', 'umidade_max_media_mensal_uf',
                   'umidade_min_media_mensal_uf']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.ravel()

for i, var in enumerate(variaveis_clima):
    axes[i].scatter(df[var], df['Quantidade de Casos'], alpha=0.5)
    axes[i].set_xlabel(var.replace('_', ' ').title())
    axes[i].set_ylabel('Casos de Dengue')
    axes[i].set_title(f'Casos vs {var.replace("_", " ").title()}')

    # Correlação
    corr = df[var].corr(df['Quantidade de Casos'])
    axes[i].text(0.05, 0.95, f'Corr: {corr:.3f}', transform=axes[i].transAxes,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Remover subplot extra
fig.delaxes(axes[5])

plt.tight_layout()
plt.show()

In [ ]:
# Matriz de correlação das variáveis numéricas
numeric_cols = df.select_dtypes(include=[np.number]).columns
correlation_matrix = df[numeric_cols].corr()

plt.figure(figsize=(14, 12))
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, mask=mask, annot=True, cmap='coolwarm',
            center=0, fmt='.2f', square=True, cbar_kws={'shrink': 0.8})
plt.title('Matriz de Correlação - Variáveis Numéricas')
plt.tight_layout()
plt.show()

# Correlações mais fortes com a variável target
target_correlations = correlation_matrix['Quantidade de Casos'].abs().sort_values(ascending=False)
print("🔗 Correlações mais fortes com Casos de Dengue:")
for var, corr in target_correlations.head(10).items():
    if var != 'Quantidade de Casos':
        print(f"   • {var}: {corr:.3f}")

In [ ]:
# Análise temporal interativa com Plotly
df_temporal = df.groupby(['Data', 'Estado'])['Quantidade de Casos'].sum().reset_index()

# Total de casos ao longo do tempo
casos_mensais = df.groupby('Data')['Quantidade de Casos'].sum().reset_index()

fig = px.line(casos_mensais, x='Data', y='Quantidade de Casos',
              title='Evolução Temporal dos Casos de Dengue no Brasil',
              labels={'Quantidade de Casos': 'Total de Casos', 'Data': 'Data'})

fig.update_layout(height=500)
fig.show()

# Heatmap temporal por estado
df_pivot = df.groupby(['Ano', 'Estado'])['Quantidade de Casos'].sum().unstack(fill_value=0)

plt.figure(figsize=(16, 10))
sns.heatmap(df_pivot.T, cmap='YlOrRd', cbar_kws={'label': 'Casos de Dengue'})
plt.title('Casos de Dengue por Estado e Ano')
plt.xlabel('Ano')
plt.ylabel('Estado')
plt.tight_layout()
plt.show()

In [ ]:
# Análise de outliers
def detect_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return outliers

outliers = detect_outliers_iqr(df, 'Quantidade de Casos')

print(f"🎯 Análise de Outliers:")
print(f"   • Total de outliers: {len(outliers)}")
print(f"   • Percentual: {(len(outliers)/len(df)*100):.2f}%")

if len(outliers) > 0:
    print(f"\n📈 Top 5 maiores outliers:")
    top_outliers = outliers.nlargest(5, 'Quantidade de Casos')[['Estado', 'Ano', 'Mês', 'Quantidade de Casos']]
    for _, row in top_outliers.iterrows():
        print(f"   • {row['Estado']} ({row['Mês']:02d}/{row['Ano']}): {row['Quantidade de Casos']:,} casos")

In [ ]:
# Resumo das descobertas
print("🎯 RESUMO DAS PRINCIPAIS DESCOBERTAS:")
print("="*50)
print(f"1. 📊 Dataset: {df.shape[0]:,} registros, {df.shape[1]} variáveis")
print(f"2. 📅 Período: {df['Ano'].min()} - {df['Ano'].max()}")
print(f"3. 🗺️ Cobertura: {df['COD_UF'].nunique()} estados brasileiros")
print(f"4. 🏆 Estado com mais casos: {casos_por_estado.index[0]}")
print(f"5. 📈 Ano com mais casos: {df.groupby('Ano')['Quantidade de Casos'].sum().idxmax()}")
print(f"6. 🌡️ Maior correlação climática: {target_correlations.drop('Quantidade de Casos').index[0]}")
print(f"7. 📊 Outliers identificados: {len(outliers)} ({(len(outliers)/len(df)*100):.1f}%)")
print(f"8. 🔍 Valores ausentes: {'Sim' if missing_df['Valores_Ausentes'].sum() > 0 else 'Não'}")

print("\n✅ Exploração inicial concluída! Próximos passos:")
print("   → Preparação e engenharia de features")
print("   → Criação de features de lag e médias móveis")
print("   → Modelagem preditiva")

In [ ]:
# Salvar versão limpa dos dados para próximas etapas
df_clean = df.copy()
df_clean.to_csv('dados_dengue_limpos.csv', index=False)
print("💾 Dados limpos salvos em 'dados_dengue_limpos.csv'")

# Salvar estatísticas descritivas
df.describe().to_csv('estatisticas_descritivas.csv')
print("📊 Estatísticas descritivas salvas em 'estatisticas_descritivas.csv'")